In [10]:
# -*- coding: utf-8 -*-
# =====================================================================================
#  0번 셀 — 라이브러리 설치
#  새 Colab T4 런타임에서 가장 먼저 한 번 실행합니다.
#  torch는 Colab 기본 설치본을 그대로 사용하며 재설치하지 않습니다(재설치 금지 규정).
# =====================================================================================
import subprocess
import sys


def _install_baseline_packages():
    packages = [
        "fastapi",
        "uvicorn",
        "langgraph",
        "pydantic>=2",
        "sentence-transformers",
        "faiss-cpu",
        "transformers",
        "accelerate",
        "bitsandbytes",
        "pypdf",
        "python-docx",
        "beautifulsoup4",
        "requests",
    ]
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *packages],
        check=True,
    )


_install_baseline_packages()
print("[0번 셀 완료] 라이브러리 설치 완료 — 다음 셀(결과기 로직)을 실행하세요.")


[0번 셀 완료] 라이브러리 설치 완료 — 다음 셀(결과기 로직)을 실행하세요.


In [11]:
# -*- coding: utf-8 -*-
"""
카카오 약관 RAG 결과기 — Baseline 구현본 (LangGraph)

스켈레톤의 모든 [STUB] 지점을 실제 로직으로 교체했다.
함수 시그니처와 스키마는 스켈레톤 원본을 그대로 유지하며, 각 함수가 필요로 하는
import는 모두 함수 본문 안에 둔다.

Baseline 구성
  로딩 requests + BeautifulSoup / 파싱 조 번호 정규식(순번 검증)
  청킹 조(article) 단위 / 임베딩 bge-m3 / 저장 FAISS IndexFlatIP
  검색 cosine top-k(k=4) / 생성 Qwen2.5-7B-Instruct 4bit
"""

from __future__ import annotations

import json
import threading
import time
from pathlib import Path
from typing import Any, Dict, List, Literal, Optional, Tuple

from langgraph.graph import END, StateGraph
from pydantic import BaseModel, ConfigDict, Field, field_validator, model_validator

# ==================
# 0. 스텁 헬퍼
# ==================

STUB_LOG: List[str] = []


def stub(stage: str, detail: str = "") -> None:
    # detail이 있으면 " — detail"을 붙이고, 없으면 stage만 사용해 한 줄 메시지를 만든다
    line = f"[STUB] {stage}" + (f" — {detail}" if detail else "")
    # 만든 메시지를 전역 로그 리스트에 누적한다
    STUB_LOG.append(line)
    # 동시에 콘솔에도 즉시 출력한다
    print(line)


# ==================
# 1. 고정 상수
# ==================

DocName = Literal[
    "카카오계정 약관",
    "카카오 위치정보 이용약관",
    "카카오 통합서비스약관",
    "카카오 통합 약관",
]

OFFICIAL_DOCUMENT_NAMES: Tuple[str, ...] = (
    "카카오계정 약관",
    "카카오 위치정보 이용약관",
    "카카오 통합서비스약관",
    "카카오 통합 약관",
)

REQUIRED_GENERATION_MODEL_FAMILY = "Qwen2.5-Instruct"


def normalize_doc_name(value: Any) -> str:
    import re
    import unicodedata

    # 입력값을 문자열로 강제 변환한 뒤 NFC 정규화(자모 결합 형태 통일)를 적용하고,
    # 정규화된 문자열에서 공백 문자를 전부 제거해 반환한다
    return re.sub(r"\s+", "", unicodedata.normalize("NFC", str(value)))


# 공식 문서명 4종을 정규화한 집합으로 미리 만들어 둔다 (매번 재계산하지 않도록 캐시)
ALLOWED_DOCS_NORM = {normalize_doc_name(d) for d in OFFICIAL_DOCUMENT_NAMES}


# ==================
# 2. 스키마 — 설정
# ==================

class DocumentSource(BaseModel):
    doc_name: DocName
    urls: List[str] = Field(default_factory=list)
    local_path: Optional[str] = None
    effective_date: str
    note: str = ""

    @model_validator(mode="after")
    def _require_source(self) -> "DocumentSource":
        # urls도 없고 local_path도 없으면 원문을 가져올 방법이 전혀 없으므로 즉시 예외를 던진다
        if not self.urls and not self.local_path:
            raise ValueError(f"[{self.doc_name}] urls 또는 local_path 중 하나는 필요합니다.")
        # 검증을 통과하면 객체 자신을 그대로 반환한다 (pydantic model_validator 관례)
        return self


class IndexConfig(BaseModel):
    sources: List[DocumentSource]
    embedding_model_name: str = "BAAI/bge-m3"
    embedding_batch_size: int = 8
    max_chunk_chars: int = 1800
    request_timeout_s: float = 20.0


class RetrievalConfig(BaseModel):
    top_k: int = 4
    query_prefix: str = ""


class GenerationConfig(BaseModel):
    model_name: str = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
    load_in_4bit: bool = True
    max_new_tokens: int = 512
    temperature: float = 0.0
    max_context_chars: int = 6000


class PipelineConfig(BaseModel):
    index: IndexConfig
    retrieval: RetrievalConfig = RetrievalConfig()
    generation: GenerationConfig = GenerationConfig()


# ==================
# 3. 스키마 — 인덱싱
# ==================

class RawDocument(BaseModel):
    doc_name: DocName
    text: str
    source_url: str
    fetched_at: str
    char_len: int


class Article(BaseModel):
    doc_name: DocName
    article_number: int
    article_title: str = ""
    body: str

    @property
    def citation(self) -> str:
        # "문서명 제N조" 형태의 머리글을 먼저 만든다
        head = f"{self.doc_name} 제{self.article_number}조"
        # 제목이 있으면 괄호를 붙이고, 없으면 머리글만 반환한다
        return f"{head}({self.article_title})" if self.article_title else head


class Chunk(BaseModel):
    chunk_id: str
    doc_name: DocName
    article_number: int
    article_title: str = ""
    text: str


class EmbeddingBundle(BaseModel):
    chunks: List[Chunk]
    vectors: List[List[float]]
    model_name: str
    dim: int


class IndexStats(BaseModel):
    n_documents: int
    n_articles: int
    n_chunks: int
    dim: int
    per_document: Dict[str, int]
    elapsed_s: float


# ==================
# 4. 스키마 — 검색 / 증강 / 생성
# ==================

class RetrievedChunk(BaseModel):
    rank: int
    score: float
    chunk: Chunk


class RetrievalOutput(BaseModel):
    question: str
    hits: List[RetrievedChunk]
    top_k: int
    elapsed_s: float


class PromptBundle(BaseModel):
    system_prompt: str
    user_prompt: str
    context_block: str
    n_context_chunks: int


class GenerationOutput(BaseModel):
    answer_text: str
    n_new_tokens: int
    elapsed_s: float


class Evidence(BaseModel):
    doc_name: DocName
    article_number: int

    def to_pair(self) -> List[Any]:
        # [문서명, 조번호] 2원소 리스트로 직렬화한다 (제출 형식과 동일)
        return [self.doc_name, int(self.article_number)]


class AnswerPayload(BaseModel):
    answer: str
    retrieved: List[Evidence] = Field(min_length=1, max_length=4)

    @field_validator("retrieved")
    @classmethod
    def _allowed_docs(cls, v: List[Evidence]) -> List[Evidence]:
        # retrieved 안의 문서명을 하나씩 검사한다
        for item in v:
            # 정규화한 문서명이 허용 목록에 없으면 즉시 예외를 던진다
            if normalize_doc_name(item.doc_name) not in ALLOWED_DOCS_NORM:
                raise ValueError(f"허용 목록 밖 문서명: {item.doc_name}")
        # 전부 통과하면 원래 리스트를 그대로 반환한다
        return v

    def to_contract(self) -> Dict[str, Any]:
        # answer는 그대로, retrieved는 각 Evidence를 [문서명, 조번호] 쌍으로 변환한 리스트로 감싼다
        return {"answer": self.answer, "retrieved": [e.to_pair() for e in self.retrieved]}


# ==================
# 5. 스키마 — 품질 결과 (골드셋 / 제출 파일)
# ==================

class GoldArticle(BaseModel):
    doc: DocName
    article: int
    citation: str


class GoldQuestion(BaseModel):
    id: str
    question: str
    ptype: str
    difficulty: str
    gold_articles: List[GoldArticle]
    key_facts: List[str]


class GoldSet(BaseModel):
    questions: List[GoldQuestion]
    meta: Dict[str, Any] = Field(default_factory=dict, alias="_meta")

    model_config = ConfigDict(populate_by_name=True)


class SubmissionAnswer(BaseModel):
    qid: str
    retrieved: List[List[Any]]
    answer: str
    error: Optional[str] = None


class SubmissionFile(BaseModel):
    team: str
    answers: List[SubmissionAnswer]
    meta: Dict[str, Any] = Field(default_factory=dict)


class ArticleScore(BaseModel):
    qid: str
    predicted: List[List[Any]]
    gold: List[List[Any]]
    hit_at_1: bool
    hit_at_k: bool
    n_gold_matched: int
    n_gold_total: int


class KeyFactScore(BaseModel):
    qid: str
    n_key_facts: int
    n_covered_auto: int
    coverage_auto: float
    per_fact: List[Dict[str, Any]]
    needs_manual_review: bool = True


class ItemReport(BaseModel):
    qid: str
    question: str
    difficulty: str
    ptype: str
    article: ArticleScore
    key_fact: KeyFactScore
    answer_text: str


class EvalReport(BaseModel):
    n_items: int
    article_hit_at_1_rate: float
    article_hit_at_k_rate: float
    key_fact_coverage_mean: float
    items: List[ItemReport]


class PerfProtocol(BaseModel):
    requests_per_run: int = 12
    concurrency: int = 2
    warmup_requests: int = 2
    repetitions: int = 3


class PerfReport(BaseModel):
    protocol: PerfProtocol
    success_rate: float
    throughput_rps: float
    p50_latency_s: Optional[float]
    p95_latency_s: Optional[float]

In [12]:
# ==================
# 6. 인덱싱 — 로딩 및 가져오기
# ==================

DEFAULT_SOURCES: List[DocumentSource] = [
    DocumentSource(
        doc_name="카카오계정 약관",
        urls=[
            "https://www.kakao.com/policy/terms?lang=ko",
            "https://qr.kakao.com/policy/terms?lang=ko",
            "https://t1.kakaocdn.net/kakaocorp/pw/policy/files/카카오계정약관.pdf",
        ],
        effective_date="2026-05-29",
        note="HTML 실패 시 PDF로 폴백. PDF 경로도 검색 시점 기준 추정값 — 실행 전 재확인 필요.",
    ),
    DocumentSource(
        doc_name="카카오 위치정보 이용약관",
        urls=[
            "https://www.kakao.com/policy/location?lang=ko",
            "https://qr.kakao.com/policy/location?lang=ko",
        ],
        effective_date="2026-07-16",
        note="HTML 경로만 확인됨. PDF 폴백 경로는 미확보.",
    ),
    DocumentSource(
        doc_name="카카오 통합서비스약관",
        urls=[
            "https://www.kakao.com/policy/terms?type=ts&lang=ko",
            "https://qr.kakao.com/policy/terms?type=ts&lang=ko",
        ],
        effective_date="2026-05-29",
        note="URL 실제 접근 가능 여부 확인 필요. PDF 폴백 경로는 미확보.",
    ),
    DocumentSource(
        doc_name="카카오 통합 약관",
        urls=[
            "https://www.kakao.com/policy/kakaoTerms?lang=ko",
        ],
        effective_date="2022-08-25",
        note="반드시 운영진 배포 공식 아카이브 링크(2022-08-25 시행본)로 교체할 것.",
    ),
]


def _load_local_file_text(local_path: str) -> Tuple[str, str]:
    from pathlib import Path

    # 노트북과 같은 위치, 또는 Colab 세션 루트(/content) 두 곳을 후보로 만든다
    candidates = [Path(local_path), Path("/content") / local_path]
    # 후보 중 실제로 존재하는 첫 번째 경로를 찾는다 (없으면 None)
    resolved = next((path for path in candidates if path.exists()), None)
    if resolved is None:
        # 아무 후보도 없으면 시도한 경로 목록을 메시지에 담아 예외를 던진다
        tried = ", ".join(str(path) for path in candidates)
        raise FileNotFoundError(f"로컬 파일을 찾을 수 없습니다. 시도한 경로: {tried}")

    # 확장자를 소문자로 통일해 분기 기준으로 사용한다
    suffix = resolved.suffix.lower()
    if suffix == ".pdf":
        from pypdf import PdfReader

        # PDF를 열어 페이지 객체 리스트를 얻는다
        reader = PdfReader(str(resolved))
        # 각 페이지의 텍스트를 추출하고(실패 시 빈 문자열), 줄바꿈으로 이어붙인다
        text = "\n".join(page.extract_text() or "" for page in reader.pages)
    elif suffix == ".docx":
        from docx import Document

        # DOCX를 열어 문서 객체를 얻는다
        document = Document(str(resolved))
        # 빈 문단은 제외하고 문단 텍스트만 줄바꿈으로 이어붙인다
        text = "\n".join(p.text for p in document.paragraphs if p.text.strip())
    else:
        # pdf, docx가 아닌 확장자는 처리 방법이 없으므로 예외를 던진다
        raise ValueError(f"지원하지 않는 로컬 파일 형식: {suffix} ({resolved})")

    # 추출한 텍스트와 실제로 읽은 경로를 튜플로 반환한다
    return text, str(resolved)


def fetch_document(source: DocumentSource, timeout_s: float = 20.0) -> RawDocument:
    import datetime
    import re

    # "제N조" 패턴 검출용 정규식 (조 구조가 있는지 판별하는 데 사용)
    article_pattern = re.compile(r"제\s*\d+\s*조")

    def finalize(raw_text: str) -> str:
        # 줄바꿈 없는 공백(&nbsp; 등)을 일반 스페이스로 치환한다
        cleaned = raw_text.replace("\u00a0", " ")
        # 줄 단위로 쪼개 각 줄의 앞뒤 공백을 제거한다
        lines = [line.strip() for line in cleaned.split("\n")]
        # 빈 줄은 제거하고 나머지를 다시 줄바꿈으로 이어붙인다
        return "\n".join(line for line in lines if line)

    if source.local_path:
        # 로컬 경로가 지정된 경우, 로컬 파일에서 원문과 실제 경로를 읽어온다
        raw_text, resolved_path = _load_local_file_text(source.local_path)
        # 읽어온 원문을 정리한다
        text = finalize(raw_text)
        if len(article_pattern.findall(text)) < 3:
            # 조 패턴이 3개 미만이면 이 문서는 조 구조를 갖추지 못한 것으로 보고 예외를 던진다
            raise RuntimeError(
                f"[{source.doc_name}] 조 구조 미검출 · 경로={resolved_path} · len={len(text)}"
            )
        # 정상이면 RawDocument로 감싸 즉시 반환한다 (URL 폴백 로직은 실행되지 않음)
        return RawDocument(
            doc_name=source.doc_name,
            text=text,
            source_url=resolved_path,
            fetched_at=datetime.datetime.now(datetime.timezone.utc).isoformat(),
            char_len=len(text),
        )

    import requests
    from bs4 import BeautifulSoup

    # 일반 브라우저처럼 보이도록 User-Agent와 언어 헤더를 지정한다
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
            "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"
        ),
        "Accept-Language": "ko-KR,ko;q=0.9",
    }
    # URL 후보별 실패 사유를 모아두는 리스트 (전부 실패했을 때 메시지로 사용)
    failures: List[str] = []

    def is_pdf_response(url: str, response: "requests.Response") -> bool:
        # 응답 헤더의 Content-Type을 소문자로 확인한다
        content_type = response.headers.get("Content-Type", "").lower()
        # Content-Type이 PDF이거나 URL 확장자가 .pdf면 PDF 응답으로 판단한다
        return "application/pdf" in content_type or url.lower().endswith(".pdf")

    # urls 리스트를 순서대로 하나씩 시도한다
    for url in source.urls:
        try:
            # GET 요청을 보낸다
            response = requests.get(url, headers=headers, timeout=timeout_s)
            # 4xx/5xx 응답이면 여기서 예외를 발생시켜 except로 넘어간다
            response.raise_for_status()

            if is_pdf_response(url, response):
                from io import BytesIO

                from pypdf import PdfReader

                # 응답 바이트를 메모리 파일처럼 다룰 수 있게 감싼 뒤 PDF로 연다
                reader = PdfReader(BytesIO(response.content))
                # 각 페이지 텍스트를 추출해 이어붙인다
                raw_text = "\n".join(page.extract_text() or "" for page in reader.pages)
            else:
                # 인코딩을 응답이 추정한 값(없으면 utf-8)으로 설정한다
                response.encoding = response.apparent_encoding or "utf-8"
                # HTML을 파서로 로드한다
                soup = BeautifulSoup(response.text, "html.parser")
                # 본문과 무관한 태그(script, style 등)를 통째로 제거한다
                for tag in soup(["script", "style", "noscript", "header", "footer", "nav"]):
                    tag.decompose()

                # 실제 약관 본문 컨테이너를 찾는다. 템플릿에 따라 클래스 조합이 다를 수 있어
                # 구체적인 선택자부터 순서대로 시도한다.
                container_selectors = [
                    "div.wrap_terms.wrap_policy",
                    "div.wrap_terms",
                ]
                content = None
                for selector in container_selectors:
                    content = soup.select_one(selector)
                    if content is not None:
                        break
                if content is None:
                    # 어떤 선택자도 없는 템플릿이면 경고를 남기고 페이지 전체로 폴백한다
                    print(
                        f"[경고][로딩] {source.doc_name} — "
                        f"본문 컨테이너({', '.join(container_selectors)})를 찾지 못해 "
                        "전체 페이지에서 추출합니다(목차 혼입 가능, span 휴리스틱에 의존)."
                    )
                    content = soup

                # 선택된 요소의 텍스트만 줄바꿈으로 구분해 추출한다
                raw_text = content.get_text(separator="\n")

            # 추출한 원문을 정리한다
            text = finalize(raw_text)

            if len(article_pattern.findall(text)) < 3:
                # 조 패턴이 3개 미만이면 이 URL은 실패로 기록하고 다음 URL을 시도한다
                failures.append(f"{url}: 조 구조 미검출(len={len(text)})")
                continue

            # 조 구조가 확인되면 RawDocument로 감싸 즉시 반환한다 (남은 URL은 시도하지 않음)
            return RawDocument(
                doc_name=source.doc_name,
                text=text,
                source_url=url,
                fetched_at=datetime.datetime.now(datetime.timezone.utc).isoformat(),
                char_len=len(text),
            )
        except Exception as exc:
            # 요청/파싱 중 어떤 예외가 나든 실패 목록에 기록하고 다음 URL로 넘어간다
            failures.append(f"{url}: {type(exc).__name__}: {exc}")

    # 모든 URL을 다 시도했는데도 성공하지 못하면 실패 사유를 모아 예외를 던진다
    raise RuntimeError(f"[{source.doc_name}] 원문 취득 실패 — " + " | ".join(failures))


def load_documents(
    sources: List[DocumentSource],
    timeout_s: float = 20.0,
) -> List[RawDocument]:
    # 결과를 담을 빈 리스트를 만든다
    documents: List[RawDocument] = []
    # 소스를 하나씩 순회한다
    for source in sources:
        # 개별 소스에서 문서를 가져온다
        document = fetch_document(source, timeout_s)
        # 어떤 문서를 어디서 몇 자 가져왔는지 로그로 남긴다
        print(
            f"[로딩] {document.doc_name} · {document.char_len}자 · {document.source_url}"
        )
        # 결과 리스트에 추가한다
        documents.append(document)
    # 전체 문서 리스트를 반환한다
    return documents


# ==================
# 7. 인덱싱 — 파싱
# ==================

def parse_articles(document: RawDocument) -> List[Article]:
    import re

    text = document.text
    # "제N조" 패턴에서 숫자를 캡처 그룹으로 뽑는 정규식
    header_pattern = re.compile(r"제\s*(\d+)\s*조")
    # "제N조" 뒤에 바로 이 접미사가 오면 본문 중 상호참조("제5조에 따른" 등)로 간주해 제외
    reference_suffix = ("에", "의", "와", "과", "및", "부터", "까지", "에서", ",", "제")

    # (시작 위치, 헤더 끝 위치, 조번호, 제목) 튜플을 모으는 리스트
    candidates: List[Tuple[int, int, int, str]] = []

    # 텍스트 전체에서 "제N조" 패턴을 순서대로 찾는다
    for match in header_pattern.finditer(text):
        # 매치된 문자열에서 조 번호를 정수로 변환한다
        number = int(match.group(1))

        # 매치 뒤 90자를 미리 잘라 제목/참조 여부 판단에 사용한다
        tail = text[match.end():match.end() + 90]
        # 앞쪽 공백을 제거한 버전을 만든다
        stripped_tail = tail.lstrip()
        if stripped_tail and stripped_tail[0] in reference_suffix:
            # 첫 글자가 참조 접미사이면 상호참조로 보고 이 매치는 건너뛴다
            continue

        # "(제목)" 형태를 먼저 시도한다
        title_match = re.match(r"[ \t]*\(([^)\n]{1,60})\)", tail)
        if title_match:
            # 괄호 안 문자열을 제목으로 쓰고, 그 뒤까지를 헤더 끝 위치로 잡는다
            title = title_match.group(1).strip()
            header_end = match.end() + title_match.end()
        else:
            # 괄호가 없으면 다음 줄바꿈 전까지(최대 60자)를 후보 제목으로 본다
            line_match = re.match(r"[ \t]*([^\n]{0,60})", tail)
            candidate = line_match.group(1).strip() if line_match else ""
            # 혹시 줄바꿈이 포함됐으면 첫 줄만 남긴다
            candidate = candidate.split("\n")[0].strip()
            # 길이가 1~40자 범위일 때만 제목으로 채택한다
            title = candidate if 0 < len(candidate) <= 40 else ""
            # 제목을 채택했으면 그 길이만큼 헤더 끝 위치를 밀고, 아니면 매치 끝 그대로 둔다
            header_end = match.end() + (len(line_match.group(0)) if title else 0)

        # 이번 매치를 후보 목록에 추가한다
        candidates.append((match.start(), header_end, number, title))

    # 1부터 연속 증가하는 구간(run)들을 담을 리스트
    runs: List[List[Tuple[int, int, int, str]]] = []
    # 현재 만들고 있는 구간
    current_run: List[Tuple[int, int, int, str]] = []
    # 다음에 나와야 할 조 번호
    expected_number = 1

    # 후보를 순서대로 확인하며 구간을 만든다
    for candidate in candidates:
        _, _, number, _ = candidate
        if number == expected_number:
            # 기대한 번호와 일치하면 현재 구간에 추가하고 기대값을 1 올린다
            current_run.append(candidate)
            expected_number += 1
        elif number == 1:
            # 기대값은 아니지만 번호가 1이면 새 구간이 시작된 것으로 보고,
            # 지금까지 만든 구간을 저장한 뒤 새 구간을 1부터 다시 연다
            if current_run:
                runs.append(current_run)
            current_run = [candidate]
            expected_number = 2
        # 기대값도 아니고 1도 아니면 잡음으로 보고 그냥 무시한다 (else 없음)

    if current_run:
        # 마지막으로 만들던 구간이 남아 있으면 저장한다
        runs.append(current_run)

    if not runs:
        # 구간이 하나도 없으면 조 구조를 전혀 못 찾은 것이므로 예외를 던진다
        raise ValueError(f"[{document.doc_name}] 조 구조 파싱 실패 — 정규식 재검토 필요")

    def run_span(run: List[Tuple[int, int, int, str]]) -> int:
        # 구간의 마지막 헤더 시작 위치에서 첫 헤더 시작 위치를 빼 폭(글자 수)을 구한다
        return run[-1][0] - run[0][0]

    # 폭이 가장 큰 구간을 실제 본문으로 채택한다 (목차는 폭이 좁음)
    headers = max(runs, key=run_span)

    if len(runs) > 1:
        # 구간이 여러 개였다면(목차 중복 가능성) 각 구간의 크기를 로그로 남긴다
        detail = ", ".join(f"{len(run)}개조/span={run_span(run)}자" for run in runs)
        print(
            f"[경고][파싱] {document.doc_name} 조 시퀀스 {len(runs)}개 발견"
            f"(목차 등 중복 가능성) — {detail} · 가장 긴 구간을 본문으로 채택"
        )

    # "제N장 ..." 형태의 장 제목 줄을 제거하기 위한 정규식
    chapter_pattern = re.compile(r"^제\s*\d+\s*장.*$", re.MULTILINE)
    articles: List[Article] = []

    # 채택된 헤더들을 순서대로 순회하며 각 조의 본문 구간을 잘라낸다
    for index, (_, header_end, number, title) in enumerate(headers):
        # 다음 헤더 시작 위치까지, 마지막 조라면 텍스트 끝까지를 본문 끝으로 삼는다
        body_end = headers[index + 1][0] if index + 1 < len(headers) else len(text)
        # 헤더 끝부터 본문 끝까지를 잘라낸다
        body = text[header_end:body_end]
        # 본문 안에 섞여 있을 수 있는 장 제목 줄을 제거한다
        body = chapter_pattern.sub("", body)
        # 연속된 줄바꿈을 하나로 줄이고 앞뒤 공백을 제거한다
        body = re.sub(r"\n{2,}", "\n", body).strip()

        # 정리된 조 정보를 Article 객체로 만들어 리스트에 추가한다
        articles.append(
            Article(
                doc_name=document.doc_name,
                article_number=number,
                article_title=title,
                body=body,
            )
        )

    if not articles:
        # 헤더는 있었지만 결과 리스트가 비었다면(이론상 발생하지 않아야 함) 예외를 던진다
        raise ValueError(f"[{document.doc_name}] 조 구조 파싱 실패 — 정규식 재검토 필요")

    # 본문이 20자 미만인 조만 골라 "제N조(len=..)" 형태 문자열 리스트로 만든다
    suspicious = [
        f"제{a.article_number}조(len={len(a.body)})"
        for a in articles
        if len(a.body) < 20
    ]
    if suspicious:
        # 의심스러운 조가 있으면 목록을 경고로 출력한다
        print(
            f"[경고][파싱] {document.doc_name} 본문이 20자 미만인 조 {len(suspicious)}건 "
            f"— 제목 추출 로직이 본문을 흡수했을 가능성: " + ", ".join(suspicious)
        )

    # 최종적으로 몇 조부터 몇 조까지 몇 개를 파싱했는지 로그로 남긴다
    print(f"[파싱] {document.doc_name} · 제1조~제{articles[-1].article_number}조 "
          f"({len(articles)}개)")
    # 완성된 조 리스트를 반환한다
    return articles


# ==================
# 8. 인덱싱 — 청킹
# ==================

def chunk_articles(articles: List[Article], config: IndexConfig) -> List[Chunk]:
    # 결과 청크를 담을 리스트
    chunks: List[Chunk] = []
    # 본문이 비어 있던 조 번호를 기록할 리스트 (경고 출력용)
    empty_body_articles: List[str] = []

    # 조를 하나씩 순회한다
    for article in articles:
        # "[문서명] 제N조" 형태의 머리글을 만든다
        header_line = f"[{article.doc_name}] 제{article.article_number}조"
        if article.article_title:
            # 제목이 있으면 괄호로 덧붙인다
            header_line += f"({article.article_title})"

        body = article.body
        if not body:
            # 본문이 비어 있으면 실패 목록에 번호를 기록하고,
            empty_body_articles.append(f"제{article.article_number}조")
            # 대신 "본문 파싱 실패" 표시가 담긴 청크 하나를 만들어 인덱스 손실을 막는다
            chunks.append(
                Chunk(
                    chunk_id=(
                        f"{normalize_doc_name(article.doc_name)}"
                        f"-{article.article_number:03d}-00"
                    ),
                    doc_name=article.doc_name,
                    article_number=article.article_number,
                    article_title=article.article_title,
                    text=f"{header_line}\n(본문 파싱 실패 — 원문 확인 필요)",
                )
            )
            # 이 조는 여기서 처리를 끝내고 다음 조로 넘어간다
            continue

        # 청크 하나에 담을 수 있는 최대 글자 수를 계산한다 (머리글 길이만큼 빼고, 최소 200자 보장)
        budget = max(200, config.max_chunk_chars - len(header_line) - 1)

        if len(body) <= budget:
            # 본문이 예산 안에 들어오면 통째로 청크 하나로 만든다
            segments = [body]
        else:
            # 예산을 넘으면 여러 조각으로 나눈다
            segments = []
            cursor = 0
            while cursor < len(body):
                # 이번 조각의 끝 위치를 예산 기준으로 잠정 결정한다
                window_end = min(cursor + budget, len(body))
                if window_end < len(body):
                    # 본문 끝이 아니라면, 자연스러운 경계(줄바꿈)를 찾아 자른다
                    boundary = body.rfind("\n", cursor + budget // 2, window_end)
                    if boundary == -1:
                        # 줄바꿈이 없으면 마침표+공백 경계를 찾는다
                        boundary = body.rfind(". ", cursor + budget // 2, window_end)
                    if boundary != -1:
                        # 경계를 찾았으면 그 위치(포함) 다음까지로 자른다
                        window_end = boundary + 1
                # 이번 조각을 잘라내 앞뒤 공백을 제거하고 리스트에 추가한다
                segments.append(body[cursor:window_end].strip())
                # 다음 조각은 이번 조각이 끝난 위치부터 시작한다
                cursor = window_end

        # 혹시 빈 조각이 섞였으면 제거하고, 전부 비었으면 빈 문자열 하나로 대체한다
        segments = [s for s in segments if s] or [""]

        # 조각마다 청크 객체를 만들어 결과 리스트에 추가한다
        for part_index, segment in enumerate(segments):
            chunks.append(
                Chunk(
                    chunk_id=(
                        f"{normalize_doc_name(article.doc_name)}"
                        f"-{article.article_number:03d}-{part_index:02d}"
                    ),
                    doc_name=article.doc_name,
                    article_number=article.article_number,
                    article_title=article.article_title,
                    text=f"{header_line}\n{segment}",
                )
            )

    if empty_body_articles:
        # 본문이 비었던 조가 있었다면 어느 조였는지 경고로 출력한다
        print(
            f"[경고][청킹] {articles[0].doc_name} 본문 파싱 실패 {len(empty_body_articles)}건: "
            + ", ".join(empty_body_articles)
        )

    # 조 개수와 최종 청크 개수를 로그로 남긴다
    print(f"[청킹] {len(articles)}개 조항 → {len(chunks)}개 청크")
    # 완성된 청크 리스트를 반환한다
    return chunks



In [13]:
# ==================
# 9. 인덱싱 — 임베딩
# ==================

class EmbedderHandle(BaseModel):
    model_config = ConfigDict(arbitrary_types_allowed=True)

    model_name: str
    dim: int
    device: str = "cpu"
    model: Any = None


def build_embedder(config: IndexConfig) -> EmbedderHandle:
    import torch
    from sentence_transformers import SentenceTransformer

    # CUDA GPU를 쓸 수 있으면 cuda, 없으면 cpu를 device로 선택한다
    device = "cuda" if torch.cuda.is_available() else "cpu"
    # 지정된 임베딩 모델을 해당 device에 로드한다
    model = SentenceTransformer(config.embedding_model_name, device=device)
    # 모델이 만드는 임베딩 벡터의 차원 수를 정수로 읽어온다
    dim = int(model.get_sentence_embedding_dimension())

    # 어떤 모델을 어느 device에 몇 차원으로 로드했는지 로그로 남긴다
    print(f"[임베딩 모델] {config.embedding_model_name} · device={device} · dim={dim}")
    # 로드한 모델과 메타정보를 핸들 객체로 감싸 반환한다
    return EmbedderHandle(
        model_name=config.embedding_model_name, dim=dim, device=device, model=model
    )


def embed_chunks(
    chunks: List[Chunk],
    embedder: EmbedderHandle,
    config: IndexConfig,
) -> EmbeddingBundle:
    # 청크 리스트에서 임베딩 대상 텍스트만 뽑아낸다
    texts = [chunk.text for chunk in chunks]
    # 배치 단위로 인코딩하고, 벡터를 L2 정규화하며(내적=cosine이 되도록), numpy 배열로 받는다
    vectors = embedder.model.encode(
        texts,
        batch_size=config.embedding_batch_size,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False,
    )

    # 몇 개 청크를 임베딩했고 결과 배열의 shape이 어떤지 로그로 남긴다
    print(f"[임베딩] {len(chunks)}개 청크 · shape={tuple(vectors.shape)}")
    # 청크 리스트와 벡터(파이썬 리스트로 변환), 모델명, 차원을 묶어 반환한다
    return EmbeddingBundle(
        chunks=chunks,
        vectors=vectors.astype("float32").tolist(),
        model_name=embedder.model_name,
        dim=int(vectors.shape[1]),
    )


# ==================
# 10. 인덱싱 — 저장
# ==================

class VectorStoreHandle(BaseModel):
    model_config = ConfigDict(arbitrary_types_allowed=True)

    backend: str = "faiss.IndexFlatIP"
    dim: int
    n_vectors: int
    chunks: List[Chunk]
    index: Any = None

    def search(self, query_vector: List[float], top_k: int) -> List[RetrievedChunk]:
        import numpy as np

        # 질의 벡터를 FAISS가 요구하는 2차원 float32 배열로 변환한다
        query = np.asarray([query_vector], dtype="float32")
        # 저장된 벡터 개수와 top_k 중 작은 값만큼 검색해 점수와 인덱스를 얻는다
        scores, indices = self.index.search(query, min(top_k, self.n_vectors))

        # 결과를 담을 리스트
        hits: List[RetrievedChunk] = []
        # 점수와 위치를 순서쌍으로 묶어 순위(1부터)와 함께 순회한다
        for rank, (score, position) in enumerate(zip(scores[0], indices[0]), start=1):
            if position < 0:
                # FAISS가 유효한 결과를 못 찾으면 -1을 반환하므로 건너뛴다
                continue
            # 순위, 점수, 해당 위치의 청크를 묶어 결과에 추가한다
            hits.append(
                RetrievedChunk(
                    rank=rank,
                    score=round(float(score), 4),
                    chunk=self.chunks[int(position)],
                )
            )
        # 검색된 청크 리스트를 반환한다
        return hits


def build_vector_store(bundle: EmbeddingBundle) -> VectorStoreHandle:
    import faiss
    import numpy as np

    # 임베딩 리스트를 float32 numpy 배열로 변환한다
    vectors = np.asarray(bundle.vectors, dtype="float32")
    # 내적 기반(코사인 유사도용) 평면 인덱스를 생성한다
    index = faiss.IndexFlatIP(bundle.dim)
    # 벡터 전체를 인덱스에 추가한다
    index.add(vectors)

    # 몇 개 벡터를 몇 차원으로 저장했는지 로그로 남긴다
    print(f"[저장] FAISS IndexFlatIP · {index.ntotal}개 벡터 · dim={bundle.dim}")
    # 인덱스와 청크 리스트, 메타정보를 핸들 객체로 감싸 반환한다
    return VectorStoreHandle(
        dim=bundle.dim,
        n_vectors=int(index.ntotal),
        chunks=bundle.chunks,
        index=index,
    )


def build_index(config: IndexConfig) -> Tuple[VectorStoreHandle, EmbedderHandle, IndexStats]:
    # 전체 인덱싱 소요 시간을 재기 위해 시작 시각을 기록한다
    started = time.perf_counter()
    # 설정된 소스들에서 원문 문서를 전부 가져온다
    documents = load_documents(config.sources, config.request_timeout_s)

    # 전체 조를 모을 리스트
    articles: List[Article] = []
    # 문서마다 조 단위로 파싱해 리스트에 이어붙인다
    for document in documents:
        articles.extend(parse_articles(document))

    # 조 리스트를 청크로 분할한다
    chunks = chunk_articles(articles, config)
    # 임베딩 모델을 로드한다
    embedder = build_embedder(config)
    # 청크를 임베딩한다
    bundle = embed_chunks(chunks, embedder, config)
    # 임베딩 결과로 벡터 저장소를 만든다
    store = build_vector_store(bundle)

    # 문서별 청크 개수를 세기 위한 딕셔너리
    per_document: Dict[str, int] = {}
    for chunk in chunks:
        # 해당 문서명의 카운트를 1 증가시킨다 (없으면 0에서 시작)
        per_document[chunk.doc_name] = per_document.get(chunk.doc_name, 0) + 1

    # 인덱싱 통계를 하나의 객체로 정리한다
    stats = IndexStats(
        n_documents=len(documents),
        n_articles=len(articles),
        n_chunks=len(chunks),
        dim=store.dim,
        per_document=per_document,
        elapsed_s=round(time.perf_counter() - started, 4),
    )
    # 벡터 저장소, 임베더, 통계를 튜플로 반환한다
    return store, embedder, stats

# ==================
# 13. 생성
# ==================

class GeneratorHandle(BaseModel):
    model_config = ConfigDict(arbitrary_types_allowed=True)

    model_name: str
    load_in_4bit: bool
    model: Any = None
    tokenizer: Any = None


def load_generator(config: GenerationConfig) -> GeneratorHandle:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    if REQUIRED_GENERATION_MODEL_FAMILY.split("-")[0] not in config.model_name:
        raise ValueError(f"생성 모델은 {REQUIRED_GENERATION_MODEL_FAMILY} 계열이어야 합니다.")

    tokenizer = AutoTokenizer.from_pretrained(config.model_name)

    # 체크포인트에 저장된 양자화 설정은 그대로 쓰되, compute dtype만
    # T4(Turing, bfloat16 네이티브 가속 없음)에 맞게 float16으로 재지정한다.
    quantization_config = BitsAndBytesConfig(bnb_4bit_compute_dtype=torch.float16)

    model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        device_map="auto",
        quantization_config=quantization_config,
    )
    model.eval()

    print(f"[생성 모델] {config.model_name} · compute_dtype=float16 재지정")
    return GeneratorHandle(
        model_name=config.model_name,
        load_in_4bit=config.load_in_4bit,
        model=model,
        tokenizer=tokenizer,
    )


def generate(prompt: PromptBundle, generator: GeneratorHandle) -> GenerationOutput:
    import torch

    # 생성 소요 시간 측정을 위해 시작 시각을 기록한다
    started = time.perf_counter()
    tokenizer = generator.tokenizer

    # 시스템/사용자 프롬프트를 채팅 메시지 형식으로 구성한다
    messages = [
        {"role": "system", "content": prompt.system_prompt},
        {"role": "user", "content": prompt.user_prompt},
    ]
    # 모델 전용 채팅 템플릿을 적용해 하나의 문자열로 만든다 (생성 시작 토큰 포함)
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    # 문자열을 토큰화하고 모델이 있는 device로 옮긴다
    model_inputs = tokenizer([text], return_tensors="pt").to(generator.model.device)

    # 그래디언트 계산 없이(추론 전용) 텍스트를 생성한다
    with torch.inference_mode():
        generated = generator.model.generate(
            **model_inputs,
            max_new_tokens=512,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        )

    # 입력 프롬프트의 토큰 길이를 구한다
    input_length = model_inputs["input_ids"].shape[1]
    # 생성 결과에서 입력 길이 이후, 즉 새로 생성된 토큰만 잘라낸다
    new_token_ids = generated[0][input_length:]
    # 새 토큰만 텍스트로 디코딩하고(특수 토큰 제거) 앞뒤 공백을 정리한다
    answer_text = tokenizer.decode(new_token_ids, skip_special_tokens=True).strip()

    # 답변 텍스트, 새로 생성된 토큰 수, 걸린 시간을 묶어 반환한다
    return GenerationOutput(
        answer_text=answer_text,
        n_new_tokens=int(new_token_ids.shape[0]),
        elapsed_s=round(time.perf_counter() - started, 4),
    )


def select_evidence(retrieval: RetrievalOutput, max_items: int = 4) -> List[Evidence]:
    # 최종 근거 리스트
    evidence: List[Evidence] = []
    # 이미 담은 (문서명, 조번호) 쌍을 기록해 중복을 막는 집합
    seen: set[Tuple[str, int]] = set()
    # 검색 결과를 순위대로 순회한다
    for hit in retrieval.hits:
        # 이 청크의 (문서명, 조번호) 키를 만든다
        key = (hit.chunk.doc_name, hit.chunk.article_number)
        if key in seen:
            # 이미 같은 조를 근거로 담았으면 건너뛴다 (같은 조가 여러 청크로 쪼개졌을 수 있으므로)
            continue
        # 처음 보는 키면 집합에 기록한다
        seen.add(key)
        # Evidence 객체로 만들어 리스트에 추가한다
        evidence.append(
            Evidence(doc_name=hit.chunk.doc_name, article_number=hit.chunk.article_number)
        )
        if len(evidence) >= max_items:
            # 최대 개수에 도달하면 더 담지 않고 멈춘다
            break
    if not evidence:
        # 검색 결과가 아예 없었다면(이론상 드묾) 최소 1개 요구사항을 맞추기 위해 기본값을 넣는다
        evidence.append(Evidence(doc_name="카카오 통합 약관", article_number=1))
    # 완성된 근거 리스트를 반환한다
    return evidence

# ==================
# 15. 부팅 + 고정 진입점
# ==================

# 기본 소스로 구성한 전역 파이프라인 설정
PIPELINE_CONFIG = PipelineConfig(index=IndexConfig(sources=DEFAULT_SOURCES))

# 아래 5개는 bootstrap()이 채우기 전까지 None으로 시작하는 전역 상태값이다
_STORE: Optional[VectorStoreHandle] = None
_EMBEDDER: Optional[EmbedderHandle] = None
_GENERATOR: Optional[GeneratorHandle] = None
_GRAPH: Optional[Any] = None
INDEX_STATS: Optional[IndexStats] = None


def bootstrap(config: PipelineConfig = PIPELINE_CONFIG) -> None:
    # 함수 안에서 전역 변수를 재할당하기 위해 global로 선언한다
    global _STORE, _EMBEDDER, _GENERATOR, _GRAPH, INDEX_STATS

    if _GRAPH is not None:
        # 이미 초기화가 끝났으면(그래프가 존재하면) 아무 것도 하지 않고 즉시 반환한다 (멱등성)
        return

    # 인덱싱을 수행해 벡터 저장소, 임베더, 통계를 전역 변수에 채운다
    _STORE, _EMBEDDER, INDEX_STATS = build_index(config.index)
    # 인덱싱 통계를 로그로 남긴다
    print(f"[인덱싱 완료] {INDEX_STATS.model_dump()}")

    # 생성 모델을 로드해 전역 변수에 채운다
    _GENERATOR = load_generator(config.generation)
    # 노드들이 공유할 컨텍스트 객체를 만든다
    ctx = PipelineContext(
        store=_STORE, embedder=_EMBEDDER, generator=_GENERATOR, config=config
    )
    # 컨텍스트를 바탕으로 그래프를 빌드해 전역 변수에 채운다
    _GRAPH = build_graph(ctx)
    # 부팅이 끝났다는 로그를 남긴다
    print("[부팅 완료] run_rag_pipeline() 호출 준비됨")

In [14]:
# ==================
# 11. 검색
# ==================

def embed_query(
    question: str,
    embedder: EmbedderHandle,
    config: RetrievalConfig,
) -> List[float]:
    # 질문 앞에 접두어(있다면)를 붙여 인코딩하고, 정규화된 벡터의 첫 번째(유일한) 결과를 꺼낸다
    vector = embedder.model.encode(
        [config.query_prefix + question],
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False,
    )[0]
    # float32 파이썬 리스트로 변환해 반환한다
    return vector.astype("float32").tolist()


def retrieve(
    question: str,
    store: VectorStoreHandle,
    embedder: EmbedderHandle,
    config: RetrievalConfig,
) -> RetrievalOutput:
    # 검색 소요 시간 측정을 위해 시작 시각을 기록한다
    started = time.perf_counter()
    # 질문을 벡터로 변환한다
    query_vector = embed_query(question, embedder, config)
    # 벡터 저장소에서 상위 top_k개를 검색한다
    hits = store.search(query_vector, config.top_k)
    # 질문, 검색 결과, top_k, 걸린 시간을 묶어 반환한다
    return RetrievalOutput(
        question=question,
        hits=hits,
        top_k=config.top_k,
        elapsed_s=round(time.perf_counter() - started, 4),
    )


# ==================
# 12. 증강
# ==================

SYSTEM_PROMPT = (
    "당신은 카카오 약관 전문 어시스턴트입니다. 제공된 약관 조문만을 근거로 답하십시오. "
    "근거에 없는 내용은 지어내지 말고, 숫자·기간·조문 번호는 근거 그대로 옮기십시오."
)

FEW_SHOT_EXAMPLES = (
    "다음은 답변 방식을 보여주는 예시입니다.\n\n"
    "[예시 근거] 제5조: 회원이 탈퇴하면 본인이 작성한 게시물은 삭제된다. 다만 제3자가\n"
    "공유하거나 댓글을 단 게시물은 삭제되지 않는다.\n"
    "[예시 질문] 탈퇴하면 제가 쓴 글이 전부 삭제되나요?\n"
    "[예시 답변] 아니오. 제3자가 공유하거나 댓글을 단 게시물은 삭제되지 않습니다. (제5조)\n\n"
    "[예시 근거] 제9조: 회사는 서비스 개선을 위해 필요한 경우 약관을 개정할 수 있다.\n"
    "[예시 질문] 약관 개정 시 회원에게 별도의 금전적 보상을 지급하나요?\n"
    "[예시 답변] 제공된 약관 조문에서 확인할 수 없습니다."
)


def build_prompt(retrieval: RetrievalOutput, config: GenerationConfig) -> PromptBundle:
    # 프롬프트에 넣을 근거 블록들을 모을 리스트
    blocks: List[str] = []
    # 지금까지 사용한 글자 수 누적값
    used_chars = 0

    # 검색된 청크를 순위대로 순회한다
    for hit in retrieval.hits:
        # "[근거 N] 문서명 / 조번호 / 본문" 형태의 블록 문자열을 만든다
        block = (
            f"[근거 {hit.rank}] 문서명: {hit.chunk.doc_name} / "
            f"조번호: 제{hit.chunk.article_number}조\n"
            f"본문: {hit.chunk.text}"
        )
        if blocks and used_chars + len(block) > config.max_context_chars:
            # 이미 근거가 하나 이상 있고, 이 블록을 더하면 예산을 넘기면 여기서 멈춘다
            break
        if not blocks and len(block) > config.max_context_chars:
            # 첫 블록부터 예산을 넘기면(근거가 하나도 없는 상태를 막기 위해) 잘라서라도 넣는다
            block = block[: config.max_context_chars]
        # 블록을 리스트에 추가한다
        blocks.append(block)
        # 누적 글자 수를 갱신한다
        used_chars += len(block)

    # 블록들을 빈 줄로 이어붙여 하나의 컨텍스트 문자열로 만든다
    context_block = "\n\n".join(blocks)
    # 컨텍스트, 질문, 답변 지시문을 합쳐 사용자 프롬프트를 완성한다
    user_prompt = (
        f"{context_block}\n\n"
        f"{FEW_SHOT_EXAMPLES}\n\n"
        f"질문: {retrieval.question}\n\n"
        "위 근거 조문만을 사용해 한국어로 답하십시오. "
        "답변 안에 근거가 된 조문 번호를 함께 밝히고, "
        "근거에서 확인되지 않는 내용은 '제공된 약관 조문에서 확인할 수 없습니다'라고 답하십시오."
    )

    # 시스템 프롬프트, 사용자 프롬프트, 컨텍스트, 사용한 블록 수를 묶어 반환한다
    return PromptBundle(
        system_prompt=SYSTEM_PROMPT,
        user_prompt=user_prompt,
        context_block=context_block,
        n_context_chunks=len(blocks),
    )

# ==================
# 14. LangGraph 오케스트레이션
# ==================

class RagState(BaseModel):
    question: str
    retrieval: Optional[RetrievalOutput] = None
    prompt: Optional[PromptBundle] = None
    generation: Optional[GenerationOutput] = None
    payload: Optional[AnswerPayload] = None


class PipelineContext(BaseModel):
    model_config = ConfigDict(arbitrary_types_allowed=True)

    store: VectorStoreHandle
    embedder: EmbedderHandle
    generator: GeneratorHandle
    config: PipelineConfig


def make_retrieve_node(ctx: PipelineContext):
    def retrieve_node(state: RagState) -> Dict[str, Any]:
        # 현재 질문으로 검색을 수행하고, 결과를 상태의 retrieval 필드 갱신값으로 반환한다
        return {"retrieval": retrieve(state.question, ctx.store, ctx.embedder, ctx.config.retrieval)}

    # 그래프에 등록할 노드 함수를 반환한다
    return retrieve_node


def make_augment_node(ctx: PipelineContext):
    def augment_node(state: RagState) -> Dict[str, Any]:
        # 이전 단계(retrieve)가 반드시 실행됐어야 한다는 전제를 검사한다
        assert state.retrieval is not None
        # 검색 결과로 프롬프트를 구성하고, 상태의 prompt 필드 갱신값으로 반환한다
        return {"prompt": build_prompt(state.retrieval, ctx.config.generation)}

    return augment_node


def make_generate_node(ctx: PipelineContext):
    def generate_node(state: RagState) -> Dict[str, Any]:
        # 이전 단계(augment)가 반드시 실행됐어야 한다는 전제를 검사한다
        assert state.prompt is not None
        # 프롬프트로 텍스트를 생성하고, 상태의 generation 필드 갱신값으로 반환한다
        return {"generation": generate(state.prompt, ctx.generator)}

    return generate_node


def make_finalize_node(ctx: PipelineContext):
    def finalize_node(state: RagState) -> Dict[str, Any]:
        # 검색과 생성이 모두 끝났다는 전제를 검사한다
        assert state.retrieval is not None and state.generation is not None
        # 생성된 답변(비어 있으면 기본 문구)과, 검색 결과에서 뽑은 근거로 최종 페이로드를 만든다
        payload = AnswerPayload(
            answer=state.generation.answer_text or "제공된 약관 조문에서 확인할 수 없습니다.",
            retrieved=select_evidence(state.retrieval, ctx.config.retrieval.top_k),
        )
        # 상태의 payload 필드 갱신값으로 반환한다
        return {"payload": payload}

    return finalize_node


def build_graph(ctx: PipelineContext):
    # RagState를 상태 타입으로 하는 그래프를 만든다
    graph = StateGraph(RagState)
    # 4개 노드를 각각 이름과 함께 등록한다
    graph.add_node("retrieve", make_retrieve_node(ctx))
    graph.add_node("augment", make_augment_node(ctx))
    graph.add_node("generate", make_generate_node(ctx))
    graph.add_node("finalize", make_finalize_node(ctx))

    # 시작 노드를 retrieve로 지정한다
    graph.set_entry_point("retrieve")
    # retrieve → augment → generate → finalize → 종료 순서로 엣지를 연결한다
    graph.add_edge("retrieve", "augment")
    graph.add_edge("augment", "generate")
    graph.add_edge("generate", "finalize")
    graph.add_edge("finalize", END)
    # 그래프를 실행 가능한 형태로 컴파일해 반환한다
    return graph.compile()

# ==================
# 15. 부팅 + 고정 진입점
# ==================

def run_rag_pipeline(question: str) -> Dict[str, Any]:
    if not isinstance(question, str) or not question.strip():
        # 질문이 문자열이 아니거나 공백뿐이면 즉시 예외를 던진다
        raise ValueError("question은 비어 있지 않은 문자열이어야 합니다.")
    if _GRAPH is None:
        # 아직 bootstrap()이 호출되지 않았으면(그래프가 없으면) 예외를 던진다
        raise RuntimeError("bootstrap()이 완료되지 않았습니다.")

    # 그래프를 질문으로 초기화한 상태로 실행해 최종 상태를 얻는다
    final_state = _GRAPH.invoke(RagState(question=question.strip()))
    # 반환 타입이 dict인지 객체인지에 따라 payload를 꺼내는 방식을 분기한다
    payload = final_state["payload"] if isinstance(final_state, dict) else final_state.payload
    if isinstance(payload, dict):
        # payload가 dict 형태로 왔으면 AnswerPayload 객체로 다시 감싼다
        payload = AnswerPayload(**payload)
    # 결과기 반환 계약 형식(dict)으로 변환해 반환한다
    return payload.to_contract()

In [15]:
# -*- coding: utf-8 -*-
# =====================================================================================
#  [학생용] 결과기 개발 기본 틀 — 1번 셀
# =====================================================================================
#  이 셀은 완성된 결과기가 아닙니다. 1번 셀에 팀별 결과기를 구현한 뒤 사용합니다.
#  결과기 코랩은 아래 두 셀을 위에서 아래로 한 번 실행할 수 있어야 합니다.
#
#    1번 셀: 팀별 결과기 구현 — 이 파일의 코드
#    2번 셀: 공개 10문항 공통 러너 — 운영진 배포본, 팀 식별자 한 줄 외 수정 금지
#
#  ┌─ 반드시 유지할 계약 ───────────────────────────────────────────────────────────┐
#  │ · answer_question(question: str) 함수 이름과 입력 형식                         │
#  │ · 반환값: {"answer": 문자열, "retrieved": [[문서명, 조번호], ...]}            │
#  │ · retrieved: 실제 답변에 사용한 근거를 관련도 순으로 1~4개                    │
#  │ · 전역 FastAPI app, GET /health, POST /answer                                 │
#  │ · Qwen2.5-Instruct 계열 생성 모델을 Colab T4에서 로컬 실행                    │
#  │ · 새 Colab T4 런타임에서 외부 준비 작업 없이 위에서 아래로 한 번 실행         │
#  └────────────────────────────────────────────────────────────────────────────────┘
#
#  ┌─ 팀이 자유롭게 구현할 부분 ─────────────────────────────────────────────────────┐
#  │ · 1번 셀 안의 결과기 구현 방식과 필요한 패키지                                 │
#  │ · answer_question 함수 내부의 처리 방식                                        │
#  │   단, 위의 고정 계약과 아래의 금지 조건은 유지해야 합니다.                     │
#  └────────────────────────────────────────────────────────────────────────────────┘
#
#  사용할 수 없는 방식
#    · Google Drive 마운트, 미리 업로드한 파일, 개인 컴퓨터 경로에 의존하는 코드
#    · 외부 생성형 LLM API, 원격 임베딩·리랭커, 원격 관리형 검색 서비스
#    · 실행 중 사람의 파일 업로드·문자 입력·버튼 클릭을 기다리는 코드
#    · torch 재설치, torch.compile
#    · 질문과 관계없이 약관 원문 전체를 매 질문의 프롬프트에 넣는 방식
#
#  주의
#    · 약관 원문을 확보하는 방법은 팀별 자유 구현입니다.
#    · 공개·비공개 답변 JSON은 2번 셀이 생성합니다. 1번 셀에서 직접 만들지 않습니다.
# =====================================================================================


# -------------------------------------------------------------------------------------
# 0. 고정 기준 — 문서명과 생성 모델 계열
# -------------------------------------------------------------------------------------
# retrieved에 기록하는 문서명은 아래 네 이름 중 하나를 그대로 사용합니다.
# 조번호는 3 또는 "제3조"처럼 채점기가 조번호를 식별할 수 있는 형태로 반환합니다.
OFFICIAL_DOCUMENT_NAMES = (
    "카카오계정 약관",
    "카카오 위치정보 이용약관",
    "카카오 통합서비스약관",
    "카카오 통합 약관",
)

# 정확한 모델 크기와 로딩 옵션은 자유지만 생성 모델은 이 계열을 사용합니다.
REQUIRED_GENERATION_MODEL_FAMILY = "Qwen2.5-Instruct"


# =====================================================================================
# 1. 팀별 자유 구현 영역 — 함수 내부를 팀 코드로 교체합니다
# =====================================================================================
# 이 영역에는 팀이 필요한 패키지 설치와 결과기 구현 코드를 작성합니다.
# 구현 방법을 제한하지 않으며, 운영진은 아래 answer_question 함수만 호출합니다.


def answer_question(question: str):
    """공통 러너가 질문마다 호출하는 고정 진입점입니다.

    함수 내부 구현은 자유지만 다음 반환 계약은 반드시 유지합니다.

    return {
        "answer": "질문에 대한 최종 답변",
        "retrieved": [
            ["카카오계정 약관", 3],
            ["카카오 통합서비스약관", 7],
        ],
    }

    retrieved에는 내부 검색 후보 전체가 아니라 실제 답변 생성에 사용한 핵심 근거를
    관련도 순으로 1~4개만 기록합니다.
    """
    # if not isinstance(question, str) or not question.strip():
    #     raise ValueError("question은 비어 있지 않은 문자열이어야 합니다.")
    # raise NotImplementedError(
    #     "answer_question()을 팀별 결과기로 구현하고, "
    #     "answer 문자열과 retrieved 1~4개를 고정 형식으로 반환하세요."
    # )

    bootstrap()  # 최초 1회만 실제 초기화, 이후는 즉시 반환
    return run_rag_pipeline(question)

bootstrap()


# =====================================================================================
# 2. 고정 FastAPI 연결 영역 — 삭제하거나 경로를 바꾸지 않습니다
# =====================================================================================
# 2번 공통 러너는 아래 app을 localhost에서 실행하고 다음 주소를 호출합니다.
#   · GET  /health : 결과기 서버 준비 여부 확인
#   · POST /answer : {"question": "..."}을 보내 answer_question() 결과 수신
#
# 팀별 결과기 로직은 위 자유 구현 영역에서 작성합니다. 이 블록은 서버 연결만 담당합니다.
# 동시 요청에서 하나의 GPU 생성 모델이 충돌하지 않도록 Lock을 사용합니다.
import subprocess
import sys
import threading


def _install_server_packages():
    """공통 러너와 연결하는 데 필요한 가벼운 서버 패키지만 설치합니다."""
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "fastapi", "uvicorn"],
        check=True,
    )


_install_server_packages()

from fastapi import FastAPI, HTTPException  # noqa: E402


app = FastAPI(title="KTB AI Performance Result Generator")
_GENERATION_LOCK = threading.Lock()


@app.get("/health")
def health():
    return {"status": "ok"}


@app.post("/answer")
def answer_api(payload: dict):
    question = payload.get("question")
    if not isinstance(question, str) or not question.strip():
        raise HTTPException(status_code=400, detail="question must be a non-empty string")
    with _GENERATION_LOCK:
        return answer_question(question.strip())


print("[1번 셀 준비] 결과기 구현을 마친 뒤 2번 공통 러너를 실행하세요.")


[로딩] 카카오계정 약관 · 11162자 · https://www.kakao.com/policy/terms?lang=ko
[로딩] 카카오 위치정보 이용약관 · 5795자 · https://www.kakao.com/policy/location?lang=ko
[로딩] 카카오 통합서비스약관 · 15059자 · https://www.kakao.com/policy/terms?type=ts&lang=ko
[로딩] 카카오 통합 약관 · 18005자 · https://www.kakao.com/policy/kakaoTerms?lang=ko
[파싱] 카카오계정 약관 · 제1조~제17조 (17개)
[파싱] 카카오 위치정보 이용약관 · 제1조~제16조 (16개)
[파싱] 카카오 통합서비스약관 · 제1조~제18조 (18개)
[파싱] 카카오 통합 약관 · 제1조~제21조 (21개)
[청킹] 72개 조항 → 77개 청크


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

/tmp/ipykernel_2672/1121792432.py:23: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dim = int(model.get_sentence_embedding_dimension())


[임베딩 모델] BAAI/bge-m3 · device=cuda · dim=1024
[임베딩] 77개 청크 · shape=(77, 1024)
[저장] FAISS IndexFlatIP · 77개 벡터 · dim=1024
[인덱싱 완료] {'n_documents': 4, 'n_articles': 72, 'n_chunks': 77, 'dim': 1024, 'per_document': {'카카오계정 약관': 17, '카카오 위치정보 이용약관': 16, '카카오 통합서비스약관': 21, '카카오 통합 약관': 23}, 'elapsed_s': 23.7007}


/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:271: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

[생성 모델] unsloth/Qwen2.5-7B-Instruct-bnb-4bit · compute_dtype=float16 재지정
[부팅 완료] run_rag_pipeline() 호출 준비됨
[1번 셀 준비] 결과기 구현을 마친 뒤 2번 공통 러너를 실행하세요.


In [16]:
# 2번 셀 — 공개 10문항 답변 파일 생성
# 이 셀은 전 팀 공통이며 _SP_TEAM 한 줄 외에는 수정하지 않습니다.
# 새 Google Colab T4 런타임에서 결과기 코드를 먼저 실행한 뒤 이 셀을 실행합니다.
#
# 사용 순서
# 1. 새 Google Colab T4 런타임에서 1번 셀 결과기 코드를 실행합니다.
# 2. 이 공통 러너를 2번 셀에 그대로 둡니다.
# 3. 맨 위 _SP_TEAM에 운영진이 알려준 숫자 팀 식별자를 입력합니다.
# 4. 생성된 answers_public_<팀>.json을 결과기 코랩 파일과 함께 제출합니다.
# 공개 문항 10개 · 실행 방식: http
# ═══════════════════════════════════════════════════════════════
#  ★ 여기 한 줄만 자기 팀으로 바꾸세요. 나머지는 손대지 마세요. ★
# ═══════════════════════════════════════════════════════════════
_SP_TEAM = "3"          # 예: "1"  ← 운영진이 알려준 팀 식별자(숫자)를 그대로 적습니다
# ═══════════════════════════════════════════════════════════════

import builtins as _sp_builtins
import json as _sp_json
import os as _sp_os_rt
import re as _sp_re
import signal as _sp_signal
import socket as _sp_socket
import sys as _sp_sys
import time as _sp_time
import traceback as _sp_traceback
import unicodedata as _sp_unicodedata
import urllib.error as _sp_urlerror
import urllib.request as _sp_urlrequest

_sp_open = _sp_builtins.open
_sp_print = _sp_builtins.print

if "_sp_real_sys_exit" in globals():
    _sp_sys.exit = _sp_real_sys_exit
    if _sp_real_exit is not None:
        _sp_builtins.exit = _sp_real_exit
    if _sp_real_quit is not None:
        _sp_builtins.quit = _sp_real_quit

_SP_OUTPUT_DIR = "/content/"
_SP_OUTPUT_PREFIX = "answers_public_"
_SP_EXPECTED_OUTPUT_PATH = ""
_SP_TEAM_ALLOWED = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz_-"
_SP_TEAM_MAX_LEN = 32
_SP_TEAM_NUMERIC_ONLY = True

def _sp_team_howto(head):
    """중단 사유 + 학생이 바로 고칠 수 있는 안내를 한 덩어리로 만든다."""
    rule = (
        "1 이상의 정수를 문자열로 입력합니다. 예: 1, 2, 17"
        if _SP_TEAM_NUMERIC_ONLY
        else "영문·숫자·밑줄(_)·하이픈(-) 1~" + str(_SP_TEAM_MAX_LEN) + "자"
    )
    return (
        head
        + "\n"
        + "\n  [고치는 법] 이 셀 맨 위 ★ 상자 안의 한 줄을 이렇게 바꾸세요."
        + '\n      _SP_TEAM = "1"      ← 운영진이 알려준 팀 식별자(숫자)를 따옴표 안에 그대로'
        + "\n  [쓸 수 있는 값] " + rule
        + "\n                 띄어쓰기와 / \\ . : 같은 경로 문자는 파일 이름을 깨뜨려 쓸 수 없습니다."
        + "\n  [왜] 결과 파일 이름이 " + _SP_OUTPUT_PREFIX + "<팀>.json 이고, 채점은 이 이름으로"
        + "\n       어느 팀 답안인지 가립니다. 비워 두면 채점 자체가 되지 않습니다."
    )

def _sp_resolve_team(value):
    """_SP_TEAM 을 검사·정리해 돌려준다. 쓸 수 없는 값이면 RuntimeError 로 즉시 중단."""
    if not isinstance(value, str) or not value.strip():
        raise RuntimeError(_sp_team_howto(
            "★ 팀 식별자(_SP_TEAM)가 비어 있어 실행을 중단했습니다. 결과 파일은 만들지 않았습니다."))
    team = value.strip()
    if _SP_TEAM_NUMERIC_ONLY and not _sp_re.fullmatch(r"[1-9][0-9]*", team):
        raise RuntimeError(_sp_team_howto(
            "★ 팀 식별자(_SP_TEAM)는 운영진이 알려준 숫자여야 합니다. 지금 값: " + repr(value)))
    if len(team) > _SP_TEAM_MAX_LEN:
        raise RuntimeError(_sp_team_howto(
            "★ 팀 식별자(_SP_TEAM)가 너무 깁니다(" + str(len(team)) + "자). 팀 이름이 아니라 짧은 식별자입니다."))
    _bad = _sp_builtins.sorted(
        _sp_builtins.set(c for c in team if c not in _SP_TEAM_ALLOWED and not ("가" <= c <= "힣")))
    if _bad:
        raise RuntimeError(_sp_team_howto(
            "★ 팀 식별자(_SP_TEAM)에 파일 이름으로 쓸 수 없는 문자가 있습니다: "
            + ", ".join(repr(c) for c in _bad) + "   (지금 값: " + repr(value) + ")"))
    return team

_SP_TEAM = _sp_resolve_team(_SP_TEAM)
if any(ord(c) > 127 for c in _SP_TEAM):
    _sp_print("[주의] 팀 식별자에 한글 등 ASCII 밖 문자가 있습니다: " + _SP_TEAM
              + " — 운영진이 알려준 식별자가 맞는지 확인하세요."
              " 한글 파일 이름은 내려받기·올리기 과정에서 자모 표현이 달라져 팀이 어긋날 수 있습니다.",
              flush=True)

_SP_OUTPUT_PATH = _SP_OUTPUT_DIR.rstrip("/") + "/" + _SP_OUTPUT_PREFIX + _SP_TEAM + ".json"
if _SP_EXPECTED_OUTPUT_PATH and (_sp_os_rt.path.basename(_SP_OUTPUT_PATH)
                                 != _sp_os_rt.path.basename(_SP_EXPECTED_OUTPUT_PATH)):
    raise RuntimeError(
        "이 셀은 " + _sp_os_rt.path.basename(_SP_EXPECTED_OUTPUT_PATH) + " 용으로 생성됐는데 "
        + _sp_os_rt.path.basename(_SP_OUTPUT_PATH) + " 로 저장하려 합니다"
        "(_SP_TEAM 을 손으로 고쳤습니까?). 다른 팀으로 돌리려면 --team 을 바꿔 셀을 다시 생성하세요."
    )
_SP_AUTO_DOWNLOAD = True
_SP_QUESTIONS_JSON = (
    "[[\"P01\", \"사업자/단체 카카오계정은 계정 정보에 등록된 담당자 몇 명이 이용할 수 있으며, 다른 사람과 공유하는 것은 허용되나요?\"], [\"P02\", \"회사가 예측하거나 통제할 수 없는 사유로 서비스가 중단된 경우, 복구가 몇 시간 이상 지연되면 회사는 공지사항에 게시하여 알리나요?\"], [\"P03\", \"카카오계정 약관에서 회사가 개별 서비스와 연동하여 카카오계정에서 제공한다고 열거한 '카카오계정 서비스'의 내용 5가지는 각각 무엇인가요?\"], [\"P04\", \"회사가 위치기반서비스의 이용을 제한하거나 중지한 때에는 이용자에게 무엇을 어떤 방법으로 알리나요?\"], [\"P05\", \"회사가 위치정보 수집·이용·제공사실 확인자료를 기록·보존하는 근거는 위치정보의 보호 및 이용 등에 관한 법률 제 몇 조 제 몇 항이며, 그 자료는 어디에 기록되어 몇 개월간 보관되나요?\"], [\"P06\", \"카카오계정이 없는 사람이 통합서비스에 가입하려면 무엇을 먼저 해야 하며, 통합서비스 이용계약은 동의·확인·승낙의 어떤 순서로 체결되나요?\"], [\"P07\", \"서비스 명칭에 '카카오'가 사용되더라도 카카오 통합서비스약관의 '통합서비스'에 포함되지 않는 서비스는 누가 제공하는 서비스이며, 약관은 그 예로 무엇을 들고 있나요?\"], [\"P08\", \"카카오 통합 약관과 세부지침(회사가 정한 서비스의 개별 이용약관·운영정책·규칙 등)의 내용이 충돌하는 경우"
    ", 본 약관이 세부지침보다 우선하여 적용되나요?\"], [\"P09\", \"이용자가 서비스 사용을 중단하거나 카카오계정 및 Daum 아이디를 탈퇴한 이후, 게시물에 관하여 회사에 부여한 라이선스의 효력은 어떻게 되나요?\"], [\"P10\", \"8세 이하의 아동 등의 생명 또는 신체 보호를 위해 보호의무자가 개인위치정보의 이용 또는 제공에 동의하려면 어떤 서류에 무엇을 첨부하여 어디에 제출해야 하며, 그 동의는 어떤 효력을 갖나요?\"]]"
)
_SP_QUESTIONS = [tuple(_x) for _x in _sp_json.loads(_SP_QUESTIONS_JSON)]
_SP_ALLOWED_DOCS = _sp_json.loads("[\"카카오계정 약관\", \"카카오 통합서비스약관\", \"카카오 통합 약관\", \"카카오 위치정보 이용약관\"]")
_SP_PER_Q_TIMEOUT_S = 120
_SP_TRANSPORT = "http"
_SP_HTTP_HOST = "127.0.0.1"
_SP_HTTP_PORT = 8765
_SP_HTTP_STARTUP_TIMEOUT_S = 30
_SP_HTTP_HEALTH_PATH = "/health"
_SP_HTTP_ANSWER_PATH = "/answer"
_SP_PERFORMANCE_REQUESTS = 12
_SP_PERFORMANCE_CONCURRENCY = 2
_SP_PERFORMANCE_REPETITIONS = 3
_SP_PERFORMANCE_WARMUP_REQUESTS = 2

_sp_fn = globals().get("answer_question")
if not callable(_sp_fn):
    raise RuntimeError(
        "팀 코드에 answer_question(question) 함수가 없습니다(규정 ②). 실행을 중단합니다."
    )

_sp_doc_warnings = []
_sp_timeouts = []
_sp_http_server = None
_sp_http_thread = None

class _SpHttpTimeout(Exception):
    """HTTP 요청 시간 초과. 품질 추출에서는 timeout_qids로 기록한다."""

def _sp_http_url(path):
    return "http://" + _SP_HTTP_HOST + ":" + str(_SP_HTTP_PORT) + path

def _sp_http_json(method, path, payload=None, timeout_s=None):
    data = None
    headers = {"Accept": "application/json"}
    if payload is not None:
        data = _sp_json.dumps(payload, ensure_ascii=False).encode("utf-8")
        headers["Content-Type"] = "application/json"
    req = _sp_urlrequest.Request(
        _sp_http_url(path), data=data, headers=headers, method=method
    )
    try:
        with _sp_urlrequest.urlopen(req, timeout=timeout_s or _SP_PER_Q_TIMEOUT_S) as resp:
            raw = resp.read().decode("utf-8")
            if resp.status != 200:
                raise RuntimeError("HTTP " + str(resp.status) + ": " + raw[:500])
    except (_sp_socket.timeout, TimeoutError) as exc:
        raise _SpHttpTimeout(str(timeout_s or _SP_PER_Q_TIMEOUT_S) + "초 안에 응답하지 않았습니다.") from exc
    except _sp_urlerror.HTTPError as exc:
        raw = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError("HTTP " + str(exc.code) + ": " + raw[:500]) from exc
    except _sp_urlerror.URLError as exc:
        if isinstance(exc.reason, (_sp_socket.timeout, TimeoutError)):
            raise _SpHttpTimeout(
                str(timeout_s or _SP_PER_Q_TIMEOUT_S) + "초 안에 응답하지 않았습니다."
            ) from exc
        raise RuntimeError("HTTP 연결 실패: " + str(exc.reason)) from exc
    try:
        return _sp_json.loads(raw)
    except _sp_json.JSONDecodeError as exc:
        raise TypeError("HTTP 응답이 JSON이 아닙니다: " + raw[:500]) from exc

def _sp_start_http_server():
    global _sp_http_server, _sp_http_thread
    _sp_app = globals().get("app")
    if _sp_app is None:
        raise RuntimeError(
            "HTTP 실행 모드에는 전역 FastAPI app과 GET /health, POST /answer가 필요합니다."
        )
    try:
        import threading as _sp_threading
        import uvicorn as _sp_uvicorn
    except ImportError as exc:
        raise RuntimeError(
            "HTTP 실행 모드에는 fastapi와 uvicorn이 필요합니다. 팀 설치 목록에 추가하세요."
        ) from exc
    _sp_config = _sp_uvicorn.Config(
        _sp_app,
        host=_SP_HTTP_HOST,
        port=_SP_HTTP_PORT,
        workers=1,
        log_level="warning",
        access_log=False,
    )
    _sp_http_server = _sp_uvicorn.Server(_sp_config)
    _sp_http_thread = _sp_threading.Thread(
        target=_sp_http_server.run, name="ktb-fastapi", daemon=True
    )
    _sp_http_thread.start()
    _sp_deadline = _sp_time.time() + _SP_HTTP_STARTUP_TIMEOUT_S
    _sp_last = None
    while _sp_time.time() < _sp_deadline:
        if not _sp_http_thread.is_alive():
            raise RuntimeError("FastAPI 서버가 준비되기 전에 종료됐습니다.")
        try:
            health = _sp_http_json("GET", _SP_HTTP_HEALTH_PATH, timeout_s=1)
            if isinstance(health, dict):
                _sp_print("[서버] FastAPI /health 준비 완료: " + _sp_http_url(_SP_HTTP_HEALTH_PATH))
                return
        except Exception as exc:
            _sp_last = exc
        _sp_time.sleep(0.2)
    _sp_stop_http_server()
    raise RuntimeError(
        "FastAPI 서버가 " + str(_SP_HTTP_STARTUP_TIMEOUT_S)
        + "초 안에 준비되지 않았습니다: " + str(_sp_last)
    )

def _sp_stop_http_server():
    if _sp_http_server is not None:
        _sp_http_server.should_exit = True
    if _sp_http_thread is not None and _sp_http_thread.is_alive():
        _sp_http_thread.join(timeout=5)

def _sp_invoke(question):
    if _SP_TRANSPORT == "http":
        return _sp_http_json(
            "POST", _SP_HTTP_ANSWER_PATH, {"question": question},
            timeout_s=_SP_PER_Q_TIMEOUT_S,
        )
    return _sp_call_with_timeout(_sp_fn, question, _SP_PER_Q_TIMEOUT_S)

if _SP_TRANSPORT == "http":
    _sp_start_http_server()

_sp_env_warnings = []
for _sp_d in ("/content/drive", "/content/gdrive", "/gdrive"):
    if _sp_os_rt.path.ismount(_sp_d):
        _sp_env_warnings.append(_sp_d + " 가 마운트되어 있습니다")
if _sp_env_warnings:
    _sp_print("", flush=True)
    _sp_print("!" * 86, flush=True)
    _sp_print("[규정 ③ 경고] 이 세션은 운영진 실행 환경과 다릅니다.", flush=True)
    for _sp_w in _sp_env_warnings:
        _sp_print("  · " + _sp_w, flush=True)
    _sp_print("  운영진은 드라이브가 연결되지 않은 새 세션에서 실행합니다. 드라이브에 둔 약관·인덱스를", flush=True)
    _sp_print("  읽고 있다면 본선에서 전량 실패합니다. 약관은 실행 중 내려받거나 셀 안에 포함하세요.", flush=True)
    _sp_print("  확인 방법: 새 노트북을 열어 코드와 이 셀만 붙여 넣고 실행해 보세요.", flush=True)
    _sp_print("!" * 86, flush=True)
    _sp_print("", flush=True)

class _SpTimeout(BaseException):
    """문항 단위 시간 초과.

    **BaseException 을 상속하는 것이 핵심이다.** 팀 코드가 `try/except Exception` 으로
    넓게 감싸는 일은 흔한데, Exception 을 상속하면 그 handler 가 시간 초과를 삼켜
    상한이 무력화된다(그대로 다음 루프를 돌며 계속 매달린다).
    """

def _sp_call_with_timeout(fn, arg, seconds):
    """SIGALRM 으로 문항 호출에 상한을 건다.

    메인 스레드가 아니거나 SIGALRM 이 없는 환경(윈도 등)에서는 signal 설정이
    실패하므로, 그때는 상한 없이 그대로 호출한다 — 상한을 못 걸었다고 해서
    채점 자체를 포기하는 편이 더 나쁘다.

    웹 Colab 셀은 IPython 이 메인 스레드에서 실행하므로 정상 동작한다.
    """
    if not seconds or seconds <= 0:
        return fn(arg)
    _sp_secs = max(1, int(seconds))     # alarm() 은 정수만 받는다. 0 은 '취소' 라 최소 1초.

    def _sp_on_alarm(signum, frame):
        raise _SpTimeout(str(_sp_secs) + "초 안에 응답하지 않았습니다.")

    try:
        _sp_prev = _sp_signal.signal(_sp_signal.SIGALRM, _sp_on_alarm)
        _sp_signal.alarm(_sp_secs)
    except (ValueError, AttributeError, OSError):
        return fn(arg)          # 상한을 걸 수 없는 환경 — 그대로 실행
    try:
        return fn(arg)
    finally:
        _sp_signal.alarm(0)
        try:
            _sp_signal.signal(_sp_signal.SIGALRM, _sp_prev)
        except Exception:
            pass

def _sp_json_safe_art(art):
    """조번호를 JSON 으로 쓸 수 있는 값으로. 표기는 최대한 원본을 살린다.

    **여기서 흡수하지 않으면 30문항을 다 돌린 뒤 파일 저장에서 터진다.**
    일부 수치 라이브러리의 정수형은 dict 도 아니고 2원소 검사도 통과하지만
    json.dump 가 거부한다. 이 값을 흡수하지 않으면
    실패 시점이 맨 끝이라 GPU 시간을 다 쓰고 결과 파일이 없는 최악의 형태가 된다.

    '제7조' 같은 문자열은 그대로 둔다 — 채점기 _art_no 가 정수로 읽는다.
    """
    if isinstance(art, bool):        # bool 은 int 의 하위형이라 먼저 걸러 낸다
        return str(art)
    if isinstance(art, (int, str)):
        return art
    try:                              # np.int64 등 정수로 볼 수 있는 것
        return int(art)
    except (TypeError, ValueError):
        return str(art)

def _sp_norm_doc(x):
    """문서명 대조용 정규화 — NFC 통일 + 공백 전부 제거.

    ⚠️ 채점기 judge_service/engine/objective.py 의 `norm_doc` 과 **같은 규칙이어야 한다.**
    러너는 Colab 셀이라 judge_service 를 import 할 수 없어 규칙을 여기에 복제해 둔다.
    한쪽만 바뀌어 어긋나면 곧바로 오탐이 난다 — 예전에 러너가 완전 일치로 대조하던 때
    '카카오계정약관'·'카카오 계정 약관' 은 실제 채점 MRR 이 1.00 인데도 규정 ④ 위반 경고를
    맞았다. 팀은 없는 문제를 고치러 다니고(자가 확인표가 n_doc_violations == 0 을 요구한다),
    정상 팀이 경고를 맞기 시작하면 아무도 경고를 안 보게 된다.
    두 구현의 일치는 submission_pipeline/tests/test_doc_name_normalization.py 가 고정한다.
    """
    return _sp_re.sub(r"\s+", "", _sp_unicodedata.normalize("NFC", str(x)))

_SP_ALLOWED_DOCS_NORM = _sp_builtins.set(_sp_norm_doc(_d) for _d in _SP_ALLOWED_DOCS)

def _sp_normalize_retrieved(qid, value):
    """retrieved 를 근거순 [[문서명, 조번호], ...] 1~4개로 정규화."""
    if not isinstance(value, (list, tuple)):
        raise TypeError(qid + ": retrieved 는 목록이어야 합니다. (실제: " + type(value).__name__ + ")")
    out = []
    for item in value:
        if isinstance(item, dict) and "doc" in item and "article_no" in item:
            doc, art = item["doc"], item["article_no"]
        elif isinstance(item, (list, tuple)) and len(item) == 2:
            doc, art = item
        else:
            raise TypeError(qid + ": retrieved 항목은 [문서명, 조번호] 2원소여야 합니다. (실제: " + repr(item) + ")")
        doc = str(doc)
        if _SP_ALLOWED_DOCS_NORM and _sp_norm_doc(doc) not in _SP_ALLOWED_DOCS_NORM:
            _sp_doc_warnings.append({"qid": qid, "doc": doc})
        out.append([doc, _sp_json_safe_art(art)])
    if not 1 <= len(out) <= 4:
        raise ValueError(
            qid + ": retrieved 는 실제 답변 근거를 관련도 순으로 1~4개 반환해야 합니다. "
            "(실제: " + str(len(out)) + "개)"
        )
    return out

_sp_answers = []
_sp_errors = []
_sp_total = len(_SP_QUESTIONS)
_sp_print(
    "\n========== " + "공개" + " " + str(_sp_total)
    + "문항 실행 · " + _SP_TEAM + "팀 ==========",
    flush=True,
)
_sp_t0 = _sp_time.time()

for _sp_i, (_sp_qid, _sp_q) in enumerate(_SP_QUESTIONS, 1):
    _sp_print("[" + str(_sp_i).zfill(2) + "/" + str(_sp_total) + "] " + _sp_qid + " 실행 중 ...", flush=True)
    _sp_started = _sp_time.time()
    try:
        _sp_out = _sp_invoke(_sp_q)
        if not isinstance(_sp_out, dict):
            raise TypeError(_sp_qid + ": answer_question() 은 딕셔너리를 반환해야 합니다. (실제: "
                            + type(_sp_out).__name__ + ")")
        _sp_retrieved = _sp_normalize_retrieved(_sp_qid, _sp_out.get("retrieved"))
        _sp_answer = _sp_out.get("answer")
        if not isinstance(_sp_answer, str):
            raise TypeError(_sp_qid + ": answer 는 문자열이어야 합니다. (실제: "
                            + type(_sp_answer).__name__ + ")")
        _sp_answers.append({"qid": _sp_qid, "retrieved": _sp_retrieved, "answer": _sp_answer})
    except (_SpTimeout, _SpHttpTimeout) as _sp_exc:  # 한 문항이 세션 전체를 잡아먹지 않도록 끊는다.
        _sp_msg = "Timeout: " + str(_sp_exc)
        _sp_timeouts.append(_sp_qid)
        _sp_errors.append({"qid": _sp_qid, "error": _sp_msg})
        _sp_answers.append({"qid": _sp_qid, "retrieved": [], "answer": "", "error": _sp_msg})
        _sp_print("[시간초과] " + _sp_qid + " — " + _sp_msg, flush=True)
    except Exception as _sp_exc:  # 한 문항 실패로 30문항 전체를 잃지 않는다.
        _sp_msg = type(_sp_exc).__name__ + ": " + str(_sp_exc)
        _sp_errors.append({"qid": _sp_qid, "error": _sp_msg})
        _sp_answers.append({"qid": _sp_qid, "retrieved": [], "answer": "", "error": _sp_msg})
        _sp_print("[오류] " + _sp_qid + " — " + _sp_msg, flush=True)
        _sp_traceback.print_exc()
    finally:
        _sp_print("      (" + str(round(_sp_time.time() - _sp_started, 1)) + "s)", flush=True)

_sp_performance = None
if _SP_TRANSPORT == "http" and _SP_PERFORMANCE_REQUESTS > 0:
    from concurrent.futures import ThreadPoolExecutor as _SpThreadPoolExecutor

    def _sp_perf_one(index):
        _qid, _question = _SP_QUESTIONS[index % len(_SP_QUESTIONS)]
        started = _sp_time.perf_counter()
        try:
            value = _sp_http_json(
                "POST", _SP_HTTP_ANSWER_PATH, {"question": _question},
                timeout_s=_SP_PER_Q_TIMEOUT_S,
            )
            ok = (
                isinstance(value, dict)
                and isinstance(value.get("answer"), str)
                and isinstance(value.get("retrieved"), (list, tuple))
            )
            return {
                "ok": ok,
                "qid": _qid,
                "latency_s": round(_sp_time.perf_counter() - started, 4),
                "error": None if ok else "invalid_schema",
            }
        except Exception as exc:
            return {
                "ok": False,
                "qid": _qid,
                "latency_s": round(_sp_time.perf_counter() - started, 4),
                "error": type(exc).__name__ + ": " + str(exc),
            }

    def _sp_percentile(values, ratio):
        if not values:
            return None
        pos = min(len(values) - 1, max(0, int((len(values) - 1) * ratio)))
        return round(values[pos], 4)

    def _sp_median(values):
        values = sorted(values)
        if not values:
            return None
        middle = len(values) // 2
        if len(values) % 2:
            return values[middle]
        return (values[middle - 1] + values[middle]) / 2

    def _sp_perf_round(n_requests, repetition):
        started = _sp_time.perf_counter()
        with _SpThreadPoolExecutor(max_workers=max(1, _SP_PERFORMANCE_CONCURRENCY)) as pool:
            rows = list(pool.map(_sp_perf_one, range(n_requests)))
        wall_s = _sp_time.perf_counter() - started
        ok_rows = [row for row in rows if row["ok"]]
        latencies = sorted(row["latency_s"] for row in ok_rows)
        return {
            "repetition": repetition,
            "transport": "http",
            "requests": n_requests,
            "concurrency": _SP_PERFORMANCE_CONCURRENCY,
            "success": len(ok_rows),
            "fail": len(rows) - len(ok_rows),
            "success_rate": round(len(ok_rows) / len(rows), 4),
            "throughput_rps": round(len(ok_rows) / wall_s, 4) if wall_s else 0.0,
            "wall_s": round(wall_s, 4),
            "p50_latency_s": _sp_percentile(latencies, 0.50),
            "p95_latency_s": _sp_percentile(latencies, 0.95),
            "errors": [row for row in rows if not row["ok"]],
        }

    _sp_warmup = None
    if _SP_PERFORMANCE_WARMUP_REQUESTS > 0:
        _sp_print(
            "[성능] 워밍업 " + str(_SP_PERFORMANCE_WARMUP_REQUESTS) + "요청 실행 중 ...",
            flush=True,
        )
        _sp_warmup = _sp_perf_round(_SP_PERFORMANCE_WARMUP_REQUESTS, 0)

    _sp_perf_samples = []
    for _sp_repetition in range(1, _SP_PERFORMANCE_REPETITIONS + 1):
        _sp_print(
            "[성능] 측정 " + str(_sp_repetition) + "/"
            + str(_SP_PERFORMANCE_REPETITIONS) + " 실행 중 ...",
            flush=True,
        )
        _sp_perf_samples.append(
            _sp_perf_round(_SP_PERFORMANCE_REQUESTS, _sp_repetition)
        )

    _sp_success_median = _sp_median([row["success"] for row in _sp_perf_samples])
    _sp_fail_median = _sp_median([row["fail"] for row in _sp_perf_samples])
    _sp_p50_values = [
        row["p50_latency_s"] for row in _sp_perf_samples
        if row["p50_latency_s"] is not None
    ]
    _sp_p95_values = [
        row["p95_latency_s"] for row in _sp_perf_samples
        if row["p95_latency_s"] is not None
    ]
    _sp_performance = {
        "version": 2,
        "transport": "http",
        "requests": _SP_PERFORMANCE_REQUESTS,
        "concurrency": _SP_PERFORMANCE_CONCURRENCY,
        "success": int(_sp_success_median),
        "fail": int(_sp_fail_median),
        "success_rate": round(_sp_median(
            [row["success_rate"] for row in _sp_perf_samples]
        ), 4),
        "throughput_rps": round(_sp_median(
            [row["throughput_rps"] for row in _sp_perf_samples]
        ), 4),
        "wall_s": round(_sp_median(
            [row["wall_s"] for row in _sp_perf_samples]
        ), 4),
        "p50_latency_s": (
            round(_sp_median(_sp_p50_values), 4) if _sp_p50_values else None
        ),
        "p95_latency_s": (
            round(_sp_median(_sp_p95_values), 4) if _sp_p95_values else None
        ),
        "errors": [
            dict(error, repetition=sample["repetition"])
            for sample in _sp_perf_samples
            for error in sample["errors"]
        ],
        "summary_method": "median",
        "protocol": {
            "requests_per_run": _SP_PERFORMANCE_REQUESTS,
            "concurrency": _SP_PERFORMANCE_CONCURRENCY,
            "warmup_requests": _SP_PERFORMANCE_WARMUP_REQUESTS,
            "repetitions": _SP_PERFORMANCE_REPETITIONS,
        },
        "samples": _sp_perf_samples,
    }
    if _sp_warmup is not None:
        _sp_performance["warmup"] = _sp_warmup
    _sp_print(
        "[성능] closed-loop 중앙값 · "
        + str(_SP_PERFORMANCE_REQUESTS) + "요청 × "
        + str(_SP_PERFORMANCE_REPETITIONS) + "회 · 동시성 "
        + str(_SP_PERFORMANCE_CONCURRENCY) + " · 대표 성공 "
        + str(_sp_performance["success"]) + " · "
        + str(_sp_performance["throughput_rps"]) + " req/s · p95 "
        + str(_sp_performance["p95_latency_s"]) + "s",
        flush=True,
    )

_sp_stop_http_server()

_sp_submission = {"team": _SP_TEAM, "answers": _sp_answers}
if _sp_doc_warnings or _sp_timeouts or _sp_env_warnings or _sp_performance:
    _sp_submission["meta"] = {"doc_name_violations": _sp_doc_warnings,
                              "timeout_qids": _sp_timeouts,
                              "env_warnings": _sp_env_warnings,
                              "transport": _SP_TRANSPORT}
    if _sp_performance:
        _sp_submission["meta"]["performance"] = _sp_performance
_sp_text = _sp_json.dumps(_sp_submission, ensure_ascii=False, indent=2, default=str)
with _sp_open(_SP_OUTPUT_PATH, "w", encoding="utf-8") as _sp_f:
    _sp_f.write(_sp_text)

_sp_print("[완료] " + str(len(_sp_answers)) + "문항 저장: " + _SP_OUTPUT_PATH
      + "  (총 " + str(round(_sp_time.time() - _sp_t0, 1)) + "s)", flush=True)
if _sp_errors:
    _sp_print("[경고] 실패 문항 " + str(len(_sp_errors)) + "건: "
          + ", ".join(_e["qid"] for _e in _sp_errors), flush=True)
if _sp_doc_warnings:
    _sp_print("[경고] 규정 ④ 위반 — 허용 목록 밖 문서명 " + str(len(_sp_doc_warnings)) + "건: "
          + ", ".join(sorted(set(_w["doc"] for _w in _sp_doc_warnings)))
          + "  → 해당 항목은 검색 점수가 0으로 채점됩니다. 허용(띄어쓰기 차이는 무관): "
          + ", ".join(_SP_ALLOWED_DOCS), flush=True)

if _SP_AUTO_DOWNLOAD:
    try:
        from google.colab import files as _sp_files
        _sp_files.download(_SP_OUTPUT_PATH)
        _sp_print("[다운로드] 브라우저 다운로드를 시작했습니다: " + _SP_OUTPUT_PATH, flush=True)
    except Exception as _sp_dl_exc:
        _sp_print("[다운로드] 자동 다운로드 실패(" + type(_sp_dl_exc).__name__ + ": " + str(_sp_dl_exc)
                  + ") — 좌측 파일 탭에서 " + _SP_OUTPUT_PATH + " 를 직접 내려받으세요.", flush=True)

_sp_print("SUBMISSION_RUNNER_DONE " + _sp_json.dumps(
    {"team": _SP_TEAM, "output_path": _SP_OUTPUT_PATH, "n_answers": len(_sp_answers),
     "n_errors": len(_sp_errors), "failed_qids": [_e["qid"] for _e in _sp_errors],
     "n_doc_violations": len(_sp_doc_warnings), "timeout_qids": _sp_timeouts,
     "env_warnings": _sp_env_warnings, "transport": _SP_TRANSPORT,
     "performance": _sp_performance},
    ensure_ascii=False), flush=True)


[서버] FastAPI /health 준비 완료: http://127.0.0.1:8765/health

========== 공개 10문항 실행 · 3팀 ==========
[01/10] P01 실행 중 ...


[transformers] Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


      (19.8s)
[02/10] P02 실행 중 ...


[transformers] Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


      (22.0s)
[03/10] P03 실행 중 ...


...

[transformers] Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hug

[성능] closed-loop 중앙값 · 12요청 × 3회 · 동시성 2 · 대표 성공 12 · 0.0391 req/s · p95 79.2941s
[완료] 10문항 저장: /content/answers_public_3.json  (총 1229.4s)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

[다운로드] 브라우저 다운로드를 시작했습니다: /content/answers_public_3.json
SUBMISSION_RUNNER_DONE {"team": "3", "output_path": "/content/answers_public_3.json", "n_answers": 10, "n_errors": 0, "failed_qids": [], "n_doc_violations": 0, "timeout_qids": [], "env_warnings": [], "transport": "http", "performance": {"version": 2, "transport": "http", "requests": 12, "concurrency": 2, "success": 12, "fail": 0, "success_rate": 1.0, "throughput_rps": 0.0391, "wall_s": 307.2501, "p50_latency_s": 42.2841, "p95_latency_s": 79.2941, "errors": [], "summary_method": "median", "protocol": {"requests_per_run": 12, "concurrency": 2, "warmup_requests": 2, "repetitions": 3}, "samples": [{"repetition": 1, "transport": "http", "requests": 12, "concurrency": 2, "success": 12, "fail": 0, "success_rate": 1.0, "throughput_rps": 0.039, "wall_s": 307.424, "p50_latency_s": 42.2841, "p95_latency_s": 79.2941, "errors": []}, {"repetition": 2, "transport": "http", "requests": 12, "concurrency": 2, "success": 12, "fail": 0, "success_rat

In [17]:
import base64
from IPython.display import HTML

with open('/content/answers_public_3.json', 'rb') as f:
    data = f.read()

b64 = base64.b64encode(data).decode()
HTML(f'<a download="answers_public_3.json" href="data:application/json;base64,{b64}">json 다운로드</a>')